# Phase 3: Alpha Research & Statistical Learning  
## Notebook 03_12 — Kalman Filter Pairs Trading

### Research question

Can a Kalman filter improve pairs-trading research by estimating a time-varying hedge ratio and producing adaptive spread signals without look-ahead bias?

This notebook follows naturally from:

```text
03_10_statistical_arbitrage_pairs
03_11_factor_orthogonalization_and_eviction
```

The previous pairs notebook used a static hedge ratio:

$$
y_t = \alpha + \beta x_t + \epsilon_t
$$

This notebook upgrades the hedge ratio to a latent time-varying state:

$$
y_t = \alpha_t + \beta_t x_t + \epsilon_t
$$

where:

$$
\begin{bmatrix}
\alpha_t\\
\beta_t
\end{bmatrix}
=
\begin{bmatrix}
\alpha_{t-1}\\
\beta_{t-1}
\end{bmatrix}
+
\eta_t
$$

The Kalman filter estimates $\alpha_t$ and $\beta_t$ sequentially using only information available up to time $t$.

It covers:

1. static hedge ratio limitations;
2. state-space model formulation;
3. Kalman prediction and update equations;
4. synthetic pair with drifting hedge ratio;
5. static OLS baseline;
6. rolling OLS baseline;
7. Kalman-filter hedge ratio;
8. innovation and spread z-score signals;
9. no-lookahead train/test backtesting;
10. transaction cost sensitivity;
11. process-noise sensitivity;
12. regime/break diagnostics;
13. comparison with false pair;
14. limitations and research extensions.

The key idea:

> Kalman filtering turns hedge-ratio estimation into an online state-estimation problem rather than a one-shot regression.

## 1. Why static hedge ratios fail

In a static pairs model:

$$
y_t = \alpha + \beta x_t + \epsilon_t
$$

the hedge ratio $\beta$ is fixed.

But in real markets, relationships drift because of:

- changing fundamentals;
- contract rolls;
- liquidity migration;
- changing index weights;
- seasonality;
- macro regimes;
- idiosyncratic shocks;
- structural breaks.

A fixed hedge ratio can create false spread signals.

Rolling OLS adapts, but it has problems:

1. arbitrary window choice;
2. noisy estimates;
3. sudden jumps when old data leaves the window;
4. no explicit uncertainty estimate.

Kalman filtering gives a smoother online update.

## 2. State-space formulation

Observation equation:

$$
y_t = H_t \theta_t + \epsilon_t
$$

where:

$$
H_t =
\begin{bmatrix}
1 & x_t
\end{bmatrix}
$$

and:

$$
\theta_t =
\begin{bmatrix}
\alpha_t\\
\beta_t
\end{bmatrix}
$$

State equation:

$$
\theta_t = \theta_{t-1} + \eta_t
$$

Noise assumptions:

$$
\epsilon_t\sim\mathcal N(0,R)
$$

$$
\eta_t\sim\mathcal N(0,Q)
$$

The process noise covariance $Q$ controls how quickly the hedge ratio can adapt.

The observation noise variance $R$ controls how much we trust each observed price relationship.

## 3. Kalman filter equations

### Prediction step

$$
\theta_{t|t-1} = \theta_{t-1|t-1}
$$

$$
P_{t|t-1} = P_{t-1|t-1} + Q
$$

### Innovation

$$
\nu_t = y_t - H_t\theta_{t|t-1}
$$

$$
S_t = H_tP_{t|t-1}H_t^\top + R
$$

### Kalman gain

$$
K_t = P_{t|t-1}H_t^\top S_t^{-1}
$$

### Update step

$$
\theta_{t|t} = \theta_{t|t-1}+K_t\nu_t
$$

$$
P_{t|t} = (I-K_tH_t)P_{t|t-1}
$$

The standardised innovation:

$$
z_t = \frac{\nu_t}{\sqrt{S_t}}
$$

can be used as a spread signal.

## 4. Trading interpretation

If:

$$
z_t > entry
$$

then $y_t$ is high relative to the Kalman-estimated relationship, so a mean-reversion trade would short the spread:

- short $y$;
- long $\beta_t x$.

If:

$$
z_t < -entry
$$

then $y_t$ is low, so a mean-reversion trade would long the spread:

- long $y$;
- short $\beta_t x$.

The position is closed when:

$$
|z_t| < exit
$$

This is a research signal, not a production strategy.

## 5. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
@dataclass(frozen=True)
class KalmanPairsConfig:
    n_days: int = 2_500
    train_fraction: float = 0.65
    validation_fraction: float = 0.15
    annualisation: int = 252
    rolling_window: int = 126
    entry_z: float = 2.0
    exit_z: float = 0.25
    q_alpha: float = 1e-6
    q_beta: float = 2e-6
    r_obs: float = 2e-4
    cost_per_turnover: float = 0.00025
    seed: int = 42


config = KalmanPairsConfig()
config

## 6. Synthetic pair with time-varying hedge ratio

We simulate:

$$
x_t
$$

as a non-stationary log price.

Then generate:

$$
y_t = \alpha_t + \beta_t x_t + s_t
$$

where:

- $\beta_t$ slowly drifts;
- $\alpha_t$ slowly drifts;
- $s_t$ is a mean-reverting spread;
- a structural break occurs partway through the sample.

We also simulate a false pair that shares a market factor but has no stable spread.

In [ ]:
def simulate_dynamic_pair(config: KalmanPairsConfig) -> tuple[pd.DataFrame, dict]:
    rng = np.random.default_rng(config.seed)
    n = config.n_days

    dates = pd.bdate_range("2015-01-01", periods=n)

    market_ret = 0.00015 + 0.010 * rng.standard_t(df=8, size=n) * np.sqrt((8 - 2) / 8)
    x_log = 4.7 + np.cumsum(market_ret + 0.003 * rng.standard_normal(n))

    alpha_true = np.zeros(n)
    beta_true = np.zeros(n)
    spread = np.zeros(n)

    alpha_true[0] = 0.15
    beta_true[0] = 1.05

    break_point = int(0.68 * n)

    for t in range(1, n):
        beta_drift = 0.00020 * rng.standard_normal()
        alpha_drift = 0.00015 * rng.standard_normal()

        beta_true[t] = beta_true[t - 1] + beta_drift
        alpha_true[t] = alpha_true[t - 1] + alpha_drift

        # Smooth structural beta shift.
        if t > break_point:
            beta_true[t] += 0.00035

        spread_vol = 0.012 if t < break_point else 0.020
        spread[t] = 0.94 * spread[t - 1] + spread_vol * rng.standard_normal()

    y_log = alpha_true + beta_true * x_log + spread

    # False pair: correlated but not cointegrated.
    false_log = 4.3 + 0.90 * np.cumsum(market_ret) + np.cumsum(0.006 * rng.standard_normal(n)) + np.linspace(0, 0.70, n)

    df = pd.DataFrame({
        "date": dates,
        "x_log": x_log,
        "y_log": y_log,
        "false_log": false_log,
        "x_price": np.exp(x_log),
        "y_price": np.exp(y_log),
        "false_price": np.exp(false_log),
        "alpha_true": alpha_true,
        "beta_true": beta_true,
        "spread_true": spread,
        "structural_break": np.arange(n) >= break_point
    })

    truth = {
        "break_point_index": break_point,
        "break_point_date": str(dates[break_point]),
        "initial_beta": float(beta_true[0]),
        "final_beta": float(beta_true[-1])
    }

    return df, truth


data, truth = simulate_dynamic_pair(config)

data.head(), truth

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data["date"], data["x_price"] / data["x_price"].iloc[0], label="x")
plt.plot(data["date"], data["y_price"] / data["y_price"].iloc[0], label="y")
plt.plot(data["date"], data["false_price"] / data["false_price"].iloc[0], label="false")
plt.axvline(pd.Timestamp(truth["break_point_date"]), linestyle="--", label="structural break")
plt.title("Synthetic Prices")
plt.xlabel("Date")
plt.ylabel("Normalised price")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(data["date"], data["beta_true"])
plt.axvline(pd.Timestamp(truth["break_point_date"]), linestyle="--")
plt.title("True Time-Varying Hedge Ratio")
plt.xlabel("Date")
plt.ylabel("beta")
plt.show()

## 7. Chronological split

We split into:

1. train: initialise static and Kalman parameters;
2. validation: sanity-check signal parameters;
3. test: final evaluation.

No future data is used to estimate the hedge ratio during the test period.

In [ ]:
n = len(data)
train_end = int(config.train_fraction * n)
validation_end = int((config.train_fraction + config.validation_fraction) * n)

train = data.iloc[:train_end].copy()
validation = data.iloc[train_end:validation_end].copy()
test = data.iloc[validation_end:].copy()

split_metadata = {
    "n_total": int(n),
    "n_train": int(len(train)),
    "n_validation": int(len(validation)),
    "n_test": int(len(test)),
    "train_start": str(train["date"].iloc[0]),
    "train_end": str(train["date"].iloc[-1]),
    "validation_start": str(validation["date"].iloc[0]),
    "validation_end": str(validation["date"].iloc[-1]),
    "test_start": str(test["date"].iloc[0]),
    "test_end": str(test["date"].iloc[-1])
}

pd.Series(split_metadata)

## 8. Static OLS hedge ratio baseline

Estimate on train:

$$
y_t = \alpha + \beta x_t + \epsilon_t
$$

Then keep $\alpha,\beta$ fixed.

This is the baseline from the static pairs notebook.

In [ ]:
def ols_hedge_ratio(df, x_col="x_log", y_col="y_log"):
    x = df[x_col].to_numpy()
    y = df[y_col].to_numpy()

    X = np.column_stack([np.ones(len(x)), x])
    beta_hat = np.linalg.pinv(X.T @ X) @ X.T @ y

    alpha = float(beta_hat[0])
    beta = float(beta_hat[1])
    spread = y - alpha - beta * x

    return alpha, beta, spread


alpha_static, beta_static, spread_static_train = ols_hedge_ratio(train)

pd.Series({
    "alpha_static": alpha_static,
    "beta_static": beta_static,
    "train_true_beta_mean": train["beta_true"].mean(),
    "test_true_beta_mean": test["beta_true"].mean()
})

## 9. Rolling OLS hedge ratio baseline

Rolling OLS adapts by re-estimating:

$$
y_t = \alpha_t + \beta_t x_t + \epsilon_t
$$

over a trailing window.

But it is window-dependent and can be noisy.

We compute rolling OLS using only past/current data.

In [ ]:
def rolling_ols_hedge_ratio(df, window, x_col="x_log", y_col="y_log"):
    alpha = np.full(len(df), np.nan)
    beta = np.full(len(df), np.nan)

    for t in range(window - 1, len(df)):
        sub = df.iloc[t-window+1:t+1]
        a, b, _ = ols_hedge_ratio(sub, x_col=x_col, y_col=y_col)
        alpha[t] = a
        beta[t] = b

    return alpha, beta


rolling_alpha, rolling_beta = rolling_ols_hedge_ratio(data, config.rolling_window)

data["rolling_alpha"] = rolling_alpha
data["rolling_beta"] = rolling_beta

data[["date", "beta_true", "rolling_beta"]].dropna().head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data["date"], data["beta_true"], label="true beta", linewidth=2)
plt.plot(data["date"], data["rolling_beta"], label="rolling OLS beta", alpha=0.8)
plt.axhline(beta_static, linestyle="--", label="static train beta")
plt.axvline(pd.Timestamp(truth["break_point_date"]), linestyle=":")
plt.title("Static vs Rolling Hedge Ratio")
plt.xlabel("Date")
plt.ylabel("Beta")
plt.legend()
plt.show()

## 10. Kalman filter implementation

State:

$$
\theta_t =
\begin{bmatrix}
\alpha_t\\
\beta_t
\end{bmatrix}
$$

Observation:

$$
y_t = [1,x_t]\theta_t+\epsilon_t
$$

State transition:

$$
\theta_t = \theta_{t-1}+\eta_t
$$

We implement online filtering only, not smoothing.

That means the estimate at time $t$ uses observations up to time $t$, not future data.

In [ ]:
def kalman_filter_dynamic_hedge(
    df,
    theta0,
    P0,
    Q,
    R,
    x_col="x_log",
    y_col="y_log"
):
    x = df[x_col].to_numpy(dtype=float)
    y = df[y_col].to_numpy(dtype=float)
    n = len(df)

    theta_pred = np.zeros((n, 2))
    theta_filt = np.zeros((n, 2))
    P_pred_store = np.zeros((n, 2, 2))
    P_filt_store = np.zeros((n, 2, 2))
    innovation = np.zeros(n)
    innovation_var = np.zeros(n)
    kalman_gain = np.zeros((n, 2))

    theta_prev = np.asarray(theta0, dtype=float)
    P_prev = np.asarray(P0, dtype=float)

    for t in range(n):
        H = np.array([[1.0, x[t]]])

        # Prediction.
        theta_prior = theta_prev.copy()
        P_prior = P_prev + Q

        y_hat = float(H @ theta_prior)
        nu = y[t] - y_hat
        S = float(H @ P_prior @ H.T + R)

        K = (P_prior @ H.T / S).reshape(2)

        theta_post = theta_prior + K * nu
        P_post = (np.eye(2) - K[:, None] @ H) @ P_prior

        theta_pred[t] = theta_prior
        theta_filt[t] = theta_post
        P_pred_store[t] = P_prior
        P_filt_store[t] = P_post
        innovation[t] = nu
        innovation_var[t] = S
        kalman_gain[t] = K

        theta_prev = theta_post
        P_prev = P_post

    out = df[["date", x_col, y_col]].copy()
    out["alpha_pred"] = theta_pred[:, 0]
    out["beta_pred"] = theta_pred[:, 1]
    out["alpha_filt"] = theta_filt[:, 0]
    out["beta_filt"] = theta_filt[:, 1]
    out["innovation"] = innovation
    out["innovation_var"] = innovation_var
    out["innovation_z"] = innovation / np.sqrt(np.maximum(innovation_var, 1e-12))
    out["kalman_gain_alpha"] = kalman_gain[:, 0]
    out["kalman_gain_beta"] = kalman_gain[:, 1]
    out["beta_uncertainty"] = np.sqrt(P_filt_store[:, 1, 1])

    return out

## 11. Initialise Kalman filter from training OLS

We initialise:

$$
\theta_0 =
\begin{bmatrix}
\hat\alpha_{OLS}\\
\hat\beta_{OLS}
\end{bmatrix}
$$

The initial state covariance $P_0$ is deliberately broad enough to let the filter adapt.

Process noise:

$$
Q=
\begin{bmatrix}
q_\alpha & 0\\
0 & q_\beta
\end{bmatrix}
$$

Observation noise:

$$
R
$$

In [ ]:
theta0 = np.array([alpha_static, beta_static])
P0 = np.diag([0.10, 0.10])
Q = np.diag([config.q_alpha, config.q_beta])
R = config.r_obs

kalman_all = kalman_filter_dynamic_hedge(
    data,
    theta0=theta0,
    P0=P0,
    Q=Q,
    R=R
)

kalman_all[["date", "alpha_filt", "beta_filt", "innovation_z"]].head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(data["date"], data["beta_true"], label="true beta", linewidth=2)
plt.plot(kalman_all["date"], kalman_all["beta_filt"], label="Kalman filtered beta", alpha=0.85)
plt.plot(data["date"], data["rolling_beta"], label="rolling OLS beta", alpha=0.55)
plt.axhline(beta_static, linestyle="--", label="static beta")
plt.axvline(pd.Timestamp(truth["break_point_date"]), linestyle=":")
plt.title("Hedge Ratio: Static OLS vs Rolling OLS vs Kalman")
plt.xlabel("Date")
plt.ylabel("Beta")
plt.legend()
plt.show()

## 12. Hedge-ratio estimation error

Because this is synthetic data, we know the true $\beta_t$.

We compare:

1. static OLS beta;
2. rolling OLS beta;
3. Kalman beta.

In real data, true beta is unknown.

In [ ]:
comparison_beta = data[["date", "beta_true", "rolling_beta"]].copy()
comparison_beta["static_beta"] = beta_static
comparison_beta["kalman_beta"] = kalman_all["beta_filt"].to_numpy()

def beta_error_summary(df, period_name, mask):
    sub = df.loc[mask].dropna()

    rows = []
    for col in ["static_beta", "rolling_beta", "kalman_beta"]:
        err = sub[col] - sub["beta_true"]
        rows.append({
            "period": period_name,
            "method": col,
            "mae": float(np.mean(np.abs(err))),
            "rmse": float(np.sqrt(np.mean(err**2))),
            "bias": float(np.mean(err))
        })

    return rows


beta_error_rows = []
beta_error_rows += beta_error_summary(comparison_beta, "train", comparison_beta["date"].isin(train["date"]))
beta_error_rows += beta_error_summary(comparison_beta, "validation", comparison_beta["date"].isin(validation["date"]))
beta_error_rows += beta_error_summary(comparison_beta, "test", comparison_beta["date"].isin(test["date"]))

beta_error_report = pd.DataFrame(beta_error_rows)

beta_error_report

## 13. Kalman innovation as spread signal

The one-step-ahead innovation is:

$$
\nu_t = y_t - H_t\theta_{t|t-1}
$$

Standardised innovation:

$$
z_t=\frac{\nu_t}{\sqrt{S_t}}
$$

This is a natural spread signal because it accounts for state uncertainty.

A large positive $z_t$ means $y_t$ is expensive relative to the predicted relationship.

A large negative $z_t$ means $y_t$ is cheap.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(kalman_all["date"], kalman_all["innovation_z"])
plt.axhline(config.entry_z, linestyle="--")
plt.axhline(-config.entry_z, linestyle="--")
plt.axhline(0, linestyle=":")
plt.axvline(pd.Timestamp(truth["break_point_date"]), linestyle=":")
plt.title("Kalman Standardised Innovation")
plt.xlabel("Date")
plt.ylabel("Innovation z-score")
plt.show()

## 14. Static and rolling spread z-scores

We compare Kalman innovation z-scores with:

1. static OLS spread z-score;
2. rolling OLS spread z-score.

The static z-score uses training spread mean/std.

The rolling z-score uses trailing spread windows.

In [ ]:
def rolling_zscore(series, window):
    s = pd.Series(series)
    mu = s.rolling(window, min_periods=max(20, window // 3)).mean()
    sd = s.rolling(window, min_periods=max(20, window // 3)).std()
    return ((s - mu) / sd.replace(0, np.nan)).to_numpy()


static_spread_all = compute_spread = data["y_log"].to_numpy() - alpha_static - beta_static * data["x_log"].to_numpy()
train_spread_mean = np.mean(static_spread_all[:train_end])
train_spread_std = np.std(static_spread_all[:train_end], ddof=1)

data["static_spread"] = static_spread_all
data["static_z"] = (data["static_spread"] - train_spread_mean) / train_spread_std

rolling_spread = data["y_log"].to_numpy() - data["rolling_alpha"].fillna(method="bfill").to_numpy() - data["rolling_beta"].fillna(method="bfill").to_numpy() * data["x_log"].to_numpy()
data["rolling_spread"] = rolling_spread
data["rolling_z"] = rolling_zscore(rolling_spread, config.rolling_window)

data["kalman_z"] = kalman_all["innovation_z"].to_numpy()

data[["date", "static_z", "rolling_z", "kalman_z"]].head()

In [ ]:
plt.figure(figsize=(12, 6))
sample = data.iloc[train_end:]
plt.plot(sample["date"], sample["static_z"], label="static z", alpha=0.6)
plt.plot(sample["date"], sample["rolling_z"], label="rolling z", alpha=0.6)
plt.plot(sample["date"], sample["kalman_z"], label="kalman innovation z", alpha=0.9)
plt.axhline(config.entry_z, linestyle="--")
plt.axhline(-config.entry_z, linestyle="--")
plt.axhline(0, linestyle=":")
plt.title("Spread Signals After Training Period")
plt.xlabel("Date")
plt.ylabel("Z-score")
plt.legend()
plt.show()

## 15. Position generation

Trading rule:

- $z_t > entry$: short spread;
- $z_t < -entry$: long spread;
- $|z_t| < exit$: flat.

Position is applied to the next day's spread change.

This avoids using same-period realised spread movement.

In [ ]:
def generate_positions(z, entry, exit):
    z = np.asarray(z, dtype=float)
    position = np.zeros(len(z))
    current = 0.0

    for t in range(len(z)):
        z_t = z[t]

        if not np.isfinite(z_t):
            position[t] = current
            continue

        if current == 0.0:
            if z_t > entry:
                current = -1.0
            elif z_t < -entry:
                current = 1.0
        else:
            if abs(z_t) < exit:
                current = 0.0

        position[t] = current

    return position


for method_col in ["static_z", "rolling_z", "kalman_z"]:
    data[f"position_{method_col}"] = generate_positions(
        data[method_col].to_numpy(),
        config.entry_z,
        config.exit_z
    )

data[[f"position_{c}" for c in ["static_z", "rolling_z", "kalman_z"]]].describe()

## 16. Backtest spread signals

For each method, spread PnL proxy:

$$
PnL_{t+1}=position_t(s_{t+1}-s_t)
$$

For Kalman, we use the innovation/spread implied by:

$$
s_t^{Kalman}=y_t-\alpha_{t|t-1}-\beta_{t|t-1}x_t
$$

Transaction cost proxy:

$$
cost_t=c|\Delta position_t|(1+|\beta_t|)
$$

This is a simplified spread PnL, not an audited portfolio accounting engine.

In [ ]:
def backtest_signal(data, signal_col, spread_col, beta_col, position_col, config, period_name, start_idx, end_idx):
    sub = data.iloc[start_idx:end_idx].copy().reset_index(drop=True)

    spread = sub[spread_col].to_numpy()
    beta = sub[beta_col].to_numpy()
    position = sub[position_col].to_numpy()

    spread_change_next = np.r_[np.diff(spread), 0.0]
    gross_pnl = position * spread_change_next

    turnover = np.abs(np.diff(np.r_[0.0, position]))
    cost = config.cost_per_turnover * turnover * (1.0 + np.abs(beta))
    net_pnl = gross_pnl - cost

    out = pd.DataFrame({
        "date": sub["date"],
        "period": period_name,
        "method": signal_col,
        "signal": sub[signal_col],
        "spread": spread,
        "beta": beta,
        "position": position,
        "spread_change_next": spread_change_next,
        "gross_pnl": gross_pnl,
        "turnover": turnover,
        "cost": cost,
        "net_pnl": net_pnl
    })

    out["cum_net_pnl"] = out["net_pnl"].cumsum()

    return out


data["kalman_spread"] = kalman_all["innovation"].to_numpy()
data["kalman_beta_for_cost"] = kalman_all["beta_filt"].to_numpy()
data["static_beta_for_cost"] = beta_static
data["rolling_beta_for_cost"] = data["rolling_beta"].fillna(beta_static)

method_specs = [
    ("static_z", "static_spread", "static_beta_for_cost", "position_static_z"),
    ("rolling_z", "rolling_spread", "rolling_beta_for_cost", "position_rolling_z"),
    ("kalman_z", "kalman_spread", "kalman_beta_for_cost", "position_kalman_z")
]

bt_frames = []

periods = [
    ("train", 0, train_end),
    ("validation", train_end, validation_end),
    ("test", validation_end, len(data))
]

for period_name, start, end in periods:
    for signal_col, spread_col, beta_col, position_col in method_specs:
        bt_frames.append(
            backtest_signal(data, signal_col, spread_col, beta_col, position_col, config, period_name, start, end)
        )

backtests = pd.concat(bt_frames, ignore_index=True)

backtests.head()

## 17. Performance comparison

Metrics:

- total net PnL;
- annualised PnL proxy;
- volatility proxy;
- Sharpe proxy;
- max drawdown;
- turnover;
- active fraction;
- total cost.

Again, this is spread-PnL research accounting, not production execution.

In [ ]:
def performance_summary(bt):
    rows = []

    for (period, method), g in bt.groupby(["period", "method"]):
        pnl = g["net_pnl"].to_numpy()
        cum = np.cumsum(pnl)
        dd = cum - np.maximum.accumulate(cum)
        active = g["position"].abs() > 0

        rows.append({
            "period": period,
            "method": method,
            "n_days": len(g),
            "total_net_pnl": float(np.sum(pnl)),
            "annualised_pnl_proxy": float(np.mean(pnl) * config.annualisation),
            "annualised_vol_proxy": float(np.std(pnl, ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float(np.mean(pnl) / np.std(pnl, ddof=1) * np.sqrt(config.annualisation)) if np.std(pnl, ddof=1) > 0 else np.nan,
            "max_drawdown_proxy": float(np.min(dd)),
            "mean_turnover": float(g["turnover"].mean()),
            "total_turnover": float(g["turnover"].sum()),
            "active_fraction": float(active.mean()),
            "hit_rate_active": float((g.loc[active, "net_pnl"] > 0).mean()) if active.any() else np.nan,
            "total_cost": float(g["cost"].sum())
        })

    return pd.DataFrame(rows)


perf = performance_summary(backtests)

perf

In [ ]:
test_bt = backtests[backtests["period"] == "test"]

plt.figure(figsize=(12, 6))
for method, g in test_bt.groupby("method"):
    plt.plot(g["date"], g["cum_net_pnl"], label=method)
plt.axhline(0, linestyle="--")
plt.title("Test Cumulative Net PnL Proxy")
plt.xlabel("Date")
plt.ylabel("Cumulative net PnL")
plt.legend()
plt.show()

## 18. Signal behaviour around structural break

A useful adaptive hedge-ratio model should respond better to changing relationships.

We inspect:

- beta estimate;
- beta uncertainty;
- Kalman innovation z-score;
- positions.

The structural break date is known only because this is synthetic data.

In [ ]:
break_date = pd.Timestamp(truth["break_point_date"])

plot_df = data[(data["date"] >= break_date - pd.Timedelta(days=250)) & (data["date"] <= break_date + pd.Timedelta(days=350))].copy()
kalman_plot = kalman_all.loc[plot_df.index]

plt.figure(figsize=(12, 6))
plt.plot(plot_df["date"], plot_df["beta_true"], label="true beta", linewidth=2)
plt.plot(plot_df["date"], kalman_plot["beta_filt"], label="Kalman beta")
plt.fill_between(
    plot_df["date"],
    kalman_plot["beta_filt"] - 2 * kalman_plot["beta_uncertainty"],
    kalman_plot["beta_filt"] + 2 * kalman_plot["beta_uncertainty"],
    alpha=0.20,
    label="±2 beta uncertainty"
)
plt.axvline(break_date, linestyle="--", label="break")
plt.title("Kalman Hedge Ratio Around Structural Break")
plt.xlabel("Date")
plt.ylabel("Beta")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(plot_df["date"], plot_df["kalman_z"], label="Kalman z")
plt.axhline(config.entry_z, linestyle="--")
plt.axhline(-config.entry_z, linestyle="--")
plt.axvline(break_date, linestyle=":")
plt.title("Kalman Innovation z-score Around Structural Break")
plt.xlabel("Date")
plt.ylabel("z")
plt.legend()
plt.show()

## 19. Process-noise sensitivity

The process noise $Q$ controls adaptability.

- Too small: beta is too rigid.
- Too large: beta chases noise.

We test several $q_\beta$ values on the validation and test periods.

In [ ]:
def run_kalman_strategy_for_qbeta(q_beta, data, config):
    Q_test = np.diag([config.q_alpha, q_beta])
    kf = kalman_filter_dynamic_hedge(
        data,
        theta0=theta0,
        P0=P0,
        Q=Q_test,
        R=config.r_obs
    )

    df = data.copy()
    df["kalman_z_tmp"] = kf["innovation_z"].to_numpy()
    df["kalman_spread_tmp"] = kf["innovation"].to_numpy()
    df["kalman_beta_tmp"] = kf["beta_filt"].to_numpy()
    df["position_tmp"] = generate_positions(df["kalman_z_tmp"], config.entry_z, config.exit_z)

    bt_frames = []
    for period_name, start, end in periods:
        bt_frames.append(
            backtest_signal(
                df,
                "kalman_z_tmp",
                "kalman_spread_tmp",
                "kalman_beta_tmp",
                "position_tmp",
                config,
                period_name,
                start,
                end
            )
        )

    bt = pd.concat(bt_frames, ignore_index=True)
    p = performance_summary(bt)
    p["q_beta"] = q_beta

    beta_err = kf["beta_filt"].to_numpy() - data["beta_true"].to_numpy()
    p["beta_rmse_full"] = np.sqrt(np.mean(beta_err**2))

    return p


q_beta_grid = [1e-8, 5e-8, 1e-7, 5e-7, 1e-6, 2e-6, 5e-6, 1e-5, 5e-5]

q_sensitivity = pd.concat([
    run_kalman_strategy_for_qbeta(qb, data, config)
    for qb in q_beta_grid
], ignore_index=True)

q_sensitivity.head()

In [ ]:
plt.figure(figsize=(10, 6))
for period, g in q_sensitivity.groupby("period"):
    plt.plot(g["q_beta"], g["sharpe_proxy"], marker="o", label=period)
plt.xscale("log")
plt.axhline(0, linestyle="--")
plt.title("Kalman Strategy Sensitivity to q_beta")
plt.xlabel("q_beta")
plt.ylabel("Sharpe proxy")
plt.legend()
plt.show()

plt.figure(figsize=(10, 6))
test_q = q_sensitivity[q_sensitivity["period"] == "test"]
plt.plot(test_q["q_beta"], test_q["beta_rmse_full"], marker="o")
plt.xscale("log")
plt.title("Beta RMSE Sensitivity to q_beta")
plt.xlabel("q_beta")
plt.ylabel("Full-sample beta RMSE")
plt.show()

## 20. Observation-noise sensitivity

Observation noise $R$ controls how much the filter trusts the price relationship.

- Lower $R$: filter reacts more to each observation.
- Higher $R$: filter is smoother and more sceptical.

We test several values.

In [ ]:
def run_kalman_strategy_for_R(R_value, data, config):
    kf = kalman_filter_dynamic_hedge(
        data,
        theta0=theta0,
        P0=P0,
        Q=Q,
        R=R_value
    )

    df = data.copy()
    df["kalman_z_tmp"] = kf["innovation_z"].to_numpy()
    df["kalman_spread_tmp"] = kf["innovation"].to_numpy()
    df["kalman_beta_tmp"] = kf["beta_filt"].to_numpy()
    df["position_tmp"] = generate_positions(df["kalman_z_tmp"], config.entry_z, config.exit_z)

    bt_frames = []
    for period_name, start, end in periods:
        bt_frames.append(
            backtest_signal(
                df,
                "kalman_z_tmp",
                "kalman_spread_tmp",
                "kalman_beta_tmp",
                "position_tmp",
                config,
                period_name,
                start,
                end
            )
        )

    bt = pd.concat(bt_frames, ignore_index=True)
    p = performance_summary(bt)
    p["R"] = R_value

    return p


R_grid = [2e-5, 5e-5, 1e-4, 2e-4, 5e-4, 1e-3, 2e-3]

R_sensitivity = pd.concat([
    run_kalman_strategy_for_R(Rv, data, config)
    for Rv in R_grid
], ignore_index=True)

R_sensitivity.head()

In [ ]:
plt.figure(figsize=(10, 6))
for period, g in R_sensitivity.groupby("period"):
    plt.plot(g["R"], g["sharpe_proxy"], marker="o", label=period)
plt.xscale("log")
plt.axhline(0, linestyle="--")
plt.title("Kalman Strategy Sensitivity to Observation Noise R")
plt.xlabel("R")
plt.ylabel("Sharpe proxy")
plt.legend()
plt.show()

## 21. Cost sensitivity

Pairs strategies can be fragile after costs.

We vary transaction-cost assumptions and compare methods.

In [ ]:
def run_backtest_with_cost(cost, data, config):
    cfg = KalmanPairsConfig(**{**asdict(config), "cost_per_turnover": cost})

    bt_frames = []
    for period_name, start, end in periods:
        for signal_col, spread_col, beta_col, position_col in method_specs:
            bt_frames.append(
                backtest_signal(data, signal_col, spread_col, beta_col, position_col, cfg, period_name, start, end)
            )

    bt = pd.concat(bt_frames, ignore_index=True)
    p = performance_summary(bt)
    p["cost_per_turnover"] = cost
    return p


cost_grid = [0.0, 0.00010, 0.00025, 0.00050, 0.00100, 0.00200]

cost_sensitivity = pd.concat([
    run_backtest_with_cost(c, data, config)
    for c in cost_grid
], ignore_index=True)

cost_sensitivity.head()

In [ ]:
plt.figure(figsize=(10, 6))
test_cost = cost_sensitivity[cost_sensitivity["period"] == "test"]
for method, g in test_cost.groupby("method"):
    plt.plot(g["cost_per_turnover"], g["sharpe_proxy"], marker="o", label=method)
plt.axhline(0, linestyle="--")
plt.title("Test Sharpe Proxy vs Cost")
plt.xlabel("Cost per turnover")
plt.ylabel("Sharpe proxy")
plt.legend()
plt.show()

## 22. False pair diagnostic

A dynamic hedge ratio can make a bad pair look temporarily tradable.

We test Kalman filtering on the false pair:

$$
false_t = \alpha_t+\beta_t x_t+\epsilon_t
$$

A flexible model can overfit a non-cointegrated relationship.

This is a major risk.

In [ ]:
# Static OLS for false pair.
alpha_false, beta_false, _ = ols_hedge_ratio(train.rename(columns={"false_log": "y_log"}), x_col="x_log", y_col="y_log")

theta0_false = np.array([alpha_false, beta_false])

false_df = data[["date", "x_log", "false_log"]].rename(columns={"false_log": "y_log"}).copy()
false_df["beta_true"] = np.nan

false_kalman = kalman_filter_dynamic_hedge(
    false_df,
    theta0=theta0_false,
    P0=P0,
    Q=Q,
    R=R,
    x_col="x_log",
    y_col="y_log"
)

false_df["kalman_z"] = false_kalman["innovation_z"].to_numpy()
false_df["kalman_spread"] = false_kalman["innovation"].to_numpy()
false_df["kalman_beta"] = false_kalman["beta_filt"].to_numpy()
false_df["position_kalman"] = generate_positions(false_df["kalman_z"], config.entry_z, config.exit_z)

false_bt_frames = []
for period_name, start, end in periods:
    false_bt_frames.append(
        backtest_signal(
            false_df,
            "kalman_z",
            "kalman_spread",
            "kalman_beta",
            "position_kalman",
            config,
            period_name,
            start,
            end
        )
    )

false_bt = pd.concat(false_bt_frames, ignore_index=True)
false_perf = performance_summary(false_bt)

false_perf

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(false_df["date"], false_df["kalman_beta"], label="false pair Kalman beta")
plt.axvline(pd.Timestamp(truth["break_point_date"]), linestyle=":")
plt.title("Kalman Beta on False Pair")
plt.xlabel("Date")
plt.ylabel("Beta")
plt.legend()
plt.show()

plt.figure(figsize=(12, 6))
for period, g in false_bt.groupby("period"):
    plt.plot(g["date"], g["cum_net_pnl"], label=period)
plt.axhline(0, linestyle="--")
plt.title("False Pair Kalman Strategy PnL")
plt.xlabel("Date")
plt.ylabel("Cumulative net PnL")
plt.legend()
plt.show()

## 23. Innovation diagnostics

If the state-space model is well specified, standardised innovations should be closer to white noise.

We inspect:

1. mean;
2. standard deviation;
3. autocorrelation;
4. extreme z-score rate.

Large autocorrelation or too many extremes indicates misspecification.

In [ ]:
def autocorrelation(x, max_lag):
    x = np.asarray(x, dtype=float)
    x = x - np.nanmean(x)
    denom = np.nansum(x**2)

    rows = []
    for lag in range(1, max_lag + 1):
        rows.append({
            "lag": lag,
            "autocorrelation": float(np.nansum(x[lag:] * x[:-lag]) / denom)
        })

    return pd.DataFrame(rows)


innovation_diag = pd.Series({
    "kalman_z_mean": data["kalman_z"].mean(),
    "kalman_z_std": data["kalman_z"].std(ddof=1),
    "kalman_z_abs_gt_2_rate": (data["kalman_z"].abs() > 2).mean(),
    "kalman_z_abs_gt_3_rate": (data["kalman_z"].abs() > 3).mean(),
    "kalman_z_autocorr_lag1": data["kalman_z"].autocorr(lag=1)
})

innovation_acf = autocorrelation(data["kalman_z"].dropna(), 30)

innovation_diag

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(data["kalman_z"].clip(-6, 6), bins=100, density=True)
plt.title("Kalman Standardised Innovation Distribution")
plt.xlabel("z")
plt.ylabel("Density")
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(innovation_acf["lag"], innovation_acf["autocorrelation"], marker="o")
plt.axhline(0, linestyle="--")
plt.title("Kalman Innovation Autocorrelation")
plt.xlabel("Lag")
plt.ylabel("Autocorrelation")
plt.show()

## 24. Practical checklist for Kalman pairs

Before trusting a Kalman pairs strategy, check:

1. **Pair validity**  
   Is there an economic reason for the pair?

2. **Cointegration / spread stability**  
   Does the relationship mean-revert?

3. **Online filtering**  
   Are you filtering, not smoothing with future data?

4. **Process noise $Q$**  
   Is beta too rigid or too noisy?

5. **Observation noise $R$**  
   Are innovations well-scaled?

6. **Break detection**  
   Does the pair fail under structural breaks?

7. **False pair test**  
   Does the filter overfit non-cointegrated pairs?

8. **Costs and turnover**  
   Does edge survive transaction costs?

9. **Parameter sensitivity**  
   Does performance depend on one lucky $Q$ or $R$?

10. **Execution realism**  
   Does the PnL model reflect actual leg execution?

11. **Portfolio risk**  
   Are multiple pairs sharing hidden factor exposure?

12. **Model governance**  
   Are pairs evicted when filter diagnostics deteriorate?

## 25. Saving outputs

The notebook saves:

1. synthetic dynamic pair data;
2. split metadata;
3. static/rolling/Kalman hedge ratio comparison;
4. beta error report;
5. spread signal table;
6. backtest table;
7. performance table;
8. Q sensitivity;
9. R sensitivity;
10. cost sensitivity;
11. false pair diagnostics;
12. innovation diagnostics;
13. manifest.

In [ ]:
output_dir = Path("data/processed/kalman_filter_pairs_trading_v1")
output_dir.mkdir(parents=True, exist_ok=True)

data_path = output_dir / "synthetic_dynamic_pair.csv"
kalman_path = output_dir / "kalman_filter_estimates.csv"
split_path = output_dir / "split_metadata.json"
beta_error_path = output_dir / "beta_error_report.csv"
signals_path = output_dir / "signals_and_positions.csv"
backtests_path = output_dir / "backtests.csv"
perf_path = output_dir / "performance_summary.csv"
q_sensitivity_path = output_dir / "q_beta_sensitivity.csv"
R_sensitivity_path = output_dir / "R_sensitivity.csv"
cost_sensitivity_path = output_dir / "cost_sensitivity.csv"
false_bt_path = output_dir / "false_pair_backtest.csv"
false_perf_path = output_dir / "false_pair_performance.csv"
innovation_acf_path = output_dir / "innovation_acf.csv"
innovation_diag_path = output_dir / "innovation_diagnostics.csv"
manifest_path = output_dir / "manifest.json"

data.to_csv(data_path, index=False)
kalman_all.to_csv(kalman_path, index=False)
Path(split_path).write_text(json.dumps(split_metadata, indent=2, default=str))
beta_error_report.to_csv(beta_error_path, index=False)
data[[
    "date", "static_z", "rolling_z", "kalman_z",
    "position_static_z", "position_rolling_z", "position_kalman_z",
    "static_spread", "rolling_spread", "kalman_spread",
    "beta_true", "rolling_beta"
]].to_csv(signals_path, index=False)
backtests.to_csv(backtests_path, index=False)
perf.to_csv(perf_path, index=False)
q_sensitivity.to_csv(q_sensitivity_path, index=False)
R_sensitivity.to_csv(R_sensitivity_path, index=False)
cost_sensitivity.to_csv(cost_sensitivity_path, index=False)
false_bt.to_csv(false_bt_path, index=False)
false_perf.to_csv(false_perf_path, index=False)
innovation_acf.to_csv(innovation_acf_path, index=False)
innovation_diag.to_frame("value").to_csv(innovation_diag_path)

manifest = {
    "dataset_name": "kalman_filter_pairs_trading_outputs",
    "schema_version": "kalman_filter_pairs_trading_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "truth": truth,
    "static_ols": {
        "alpha_static": alpha_static,
        "beta_static": beta_static
    },
    "kalman_parameters": {
        "theta0": theta0.tolist(),
        "P0": P0.tolist(),
        "Q": Q.tolist(),
        "R": R
    },
    "performance_summary": perf.to_dict(orient="records"),
    "false_pair_performance": false_perf.to_dict(orient="records"),
    "innovation_diagnostics": innovation_diag.to_dict(),
    "notes": [
        "Synthetic pair has a drifting hedge ratio and structural break.",
        "Static OLS, rolling OLS, and Kalman hedge ratios are compared.",
        "Kalman filter is online filtering, not smoothing.",
        "Trading signal uses standardised one-step innovation.",
        "Backtest uses simplified spread PnL and turnover cost proxy.",
        "False pair diagnostic shows risk of overfitting a non-stationary relationship.",
        "This notebook is research infrastructure, not a production execution simulator."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

data_path, kalman_path, perf_path, false_perf_path, manifest_path

## 26. Limitations

### 26.1 Synthetic data

The notebook uses synthetic data where the true hedge ratio is known.

Real data does not reveal true beta.

### 26.2 Simplified state model

The state follows a random walk.

Real hedge ratios may mean-revert, jump, or change with regimes.

### 26.3 Gaussian assumptions

The Kalman filter assumes Gaussian noise.

Financial spreads can have heavy tails and jumps.

### 26.4 Fixed Q and R

Process and observation noise are fixed.

In production, these may be estimated, tuned, or made regime-dependent.

### 26.5 Simplified PnL

The backtest uses spread changes and simple turnover cost.

Real pairs execution requires:

- two-leg fills;
- slippage;
- borrow/funding;
- margin;
- execution timing;
- beta/dollar neutrality;
- contract multipliers.

### 26.6 False confidence from adaptivity

A flexible beta can hide a broken relationship.

Kalman filtering does not guarantee cointegration or tradability.

### 26.7 No portfolio construction

The notebook studies one pair and one false pair.

A real stat-arb book needs capital allocation and pair correlation control.

## 27. Research frontier and extensions

Important extensions include:

1. **Kalman filter with mean-reverting beta**  
   Use AR(1) state dynamics.

2. **Robust Kalman filter**  
   Heavy-tailed observation noise.

3. **Unscented or particle filters**  
   Nonlinear pairs relationships.

4. **EM estimation of Q and R**  
   Learn noise parameters from data.

5. **Regime-switching Kalman filter**  
   Combine HMM regimes with dynamic hedge ratios.

6. **Dynamic cointegration baskets**  
   Extend from pairs to multi-asset spreads.

7. **Execution-aware pairs backtesting**  
   Include two-leg slippage, liquidity, borrow, and margin.

8. **Break detection**  
   Evict pairs when innovation diagnostics deteriorate.

9. **Chinese futures calendar spreads**  
   Apply dynamic hedge ratios to main/deferred contracts.

10. **Cross-commodity relative value**  
   Kalman filter hedge ratios across related commodity futures.

## 28. Suggested follow-up notebooks

This notebook naturally leads to:

1. **03_13_information_coefficient_analysis**  
   Evaluate spread signals as alpha features.

2. **04_04_dynamic_covariance_and_correlation**  
   Model changing relationships across assets.

3. **05_01_vectorized_backtest_engine**  
   Generalise backtesting infrastructure.

4. **05_03_transaction_costs_slippage_latency**  
   Improve two-leg execution realism.

5. **07_01_multi_asset_cta_research_pipeline**  
   Add dynamic relative-value overlays.

6. **Chinese futures relative-value capstone**  
   Apply Kalman spreads to commodity futures pairs and calendar spreads.

## 29. Summary

This notebook implemented Kalman filter pairs trading.

It showed how to:

1. simulate a pair with drifting hedge ratio;
2. estimate a static OLS hedge ratio;
3. estimate a rolling OLS hedge ratio;
4. implement an online Kalman filter;
5. compute standardised innovations;
6. build spread z-score signals;
7. backtest static, rolling, and Kalman methods;
8. compare hedge-ratio estimation errors;
9. analyse structural break behaviour;
10. test process-noise sensitivity;
11. test observation-noise sensitivity;
12. test transaction-cost sensitivity;
13. demonstrate false-pair overfitting risk;
14. save diagnostics and manifests.

The key computational takeaway:

> Kalman filtering estimates a dynamic hedge ratio online, with uncertainty, rather than forcing a single static beta.

The key financial takeaway:

> Dynamic hedge ratios help with drifting relationships, but they do not rescue a pair that is economically broken or too costly to trade.

## 30. Further reading

- Kalman, R. E. "A New Approach to Linear Filtering and Prediction Problems."
- Harvey, *Forecasting, Structural Time Series Models and the Kalman Filter.*
- Durbin and Koopman, *Time Series Analysis by State Space Methods.*
- Vidyamurthy, *Pairs Trading.*
- Literature on dynamic hedge ratios, state-space models, Kalman-filter pairs trading, and statistical arbitrage.